# QA Predictions - Tous les Modèles

Notebook de test pour comparer les prédictions de tous les modèles disponibles:
- BiDAF V3 / V4 / V5 / V6 (from-scratch)
- BERT fine-tuné (`bert_qa_best`)

Le notebook:
1. charge les checkpoints trouvés,
2. exécute des prédictions sur un sous-ensemble dev,
3. affiche un comparatif EM/F1,
4. permet un test manuel sur un contexte + question.



In [62]:
import os
import re
import json
import math
from collections import Counter

import numpy as np
import torch

from models import QAModelV3, QAModelV5, QAModelV6
from transformers import AutoTokenizer, AutoModelForQuestionAnswering


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)



Device: mps


In [63]:
# -----------------------------
# Paramètres + utilitaires
# -----------------------------
DATA_DIR = "data"
TRAIN_PATH = os.path.join(DATA_DIR, "train-v2.0.json")
DEV_PATH = os.path.join(DATA_DIR, "dev-v2.0.json")

MAX_CONTEXT_LEN = 220
MAX_QUESTION_LEN = 40
MIN_FREQ = 2
EVAL_LIMIT = 400  # augmente si tu veux une éval plus stable

TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]")


def tokenize_advanced(text):
    return TOKEN_PATTERN.findall(text.lower())


def build_vocab(examples, min_freq=2):
    counter = Counter()
    for ex in examples:
        counter.update(ex["context_tokens"])
        counter.update(ex["question_tokens"])

    vocab = {"<PAD>": 0, "<UNK>": 1, "<CLS>": 2}
    for tok, freq in counter.items():
        if freq >= min_freq and tok not in vocab:
            vocab[tok] = len(vocab)
    return vocab


def encode(tokens, vocab_obj, max_len, add_cls=False, vocab_limit=None):
    if add_cls:
        tokens = ["<CLS>"] + tokens

    unk_id = vocab_obj["<UNK>"]
    pad_id = vocab_obj["<PAD>"]

    ids = []
    for t in tokens[:max_len]:
        idx = vocab_obj.get(t, unk_id)
        if vocab_limit is not None and idx >= vocab_limit:
            idx = unk_id
        ids.append(idx)

    return ids + [pad_id] * (max_len - len(ids))


def clean_prediction(prediction):
    prediction = re.sub(r"^[\s\(\)\[\],;:\"']+", "", prediction)
    prediction = re.sub(r"[\s\(\)\[\],;:\"']+$", "", prediction)
    prediction = re.sub(r"\s+", " ", prediction).strip()
    return prediction


def normalize_answer(s):
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def compute_em(pred, truth):
    return int(normalize_answer(pred) == normalize_answer(truth))


def compute_f1(pred, truth):
    p = normalize_answer(pred).split()
    t = normalize_answer(truth).split()
    if len(p) == 0 and len(t) == 0:
        return 1.0
    if len(p) == 0 or len(t) == 0:
        return 0.0

    common = Counter(p) & Counter(t)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(p)
    recall = num_same / len(t)
    return 2 * precision * recall / (precision + recall)


def load_train_for_vocab(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = []
    for article in data["data"]:
        for p in article["paragraphs"]:
            c_tok = tokenize_advanced(p["context"])
            for qa in p["qas"]:
                q_tok = tokenize_advanced(qa["question"])
                out.append({"context_tokens": c_tok, "question_tokens": q_tok})
    return out


def load_dev_examples(path, limit=None):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = []
    for article in data["data"]:
        for p in article["paragraphs"]:
            context = p["context"]
            context_tokens = tokenize_advanced(context)
            for qa in p["qas"]:
                question = qa["question"]
                question_tokens = tokenize_advanced(question)
                truth = qa["answers"][0]["text"] if qa.get("answers") else ""
                out.append({
                    "context": context,
                    "question": question,
                    "context_tokens": context_tokens,
                    "question_tokens": question_tokens,
                    "truth": truth,
                })
                if limit is not None and len(out) >= limit:
                    return out
    return out


print("Chargement train (vocab)...")
train_for_vocab = load_train_for_vocab(TRAIN_PATH)
vocab = build_vocab(train_for_vocab, MIN_FREQ)
VOCAB_SIZE = len(vocab)
print("Vocab size (reconstruit):", VOCAB_SIZE)

print("Chargement dev (eval)...")
dev_examples = load_dev_examples(DEV_PATH, limit=EVAL_LIMIT)
print("Dev examples:", len(dev_examples))



Chargement train (vocab)...
Vocab size (reconstruit): 80407
Chargement dev (eval)...
Dev examples: 400


In [64]:
# -----------------------------
# Prédicteurs from-scratch (V3/V4/V5/V6)
# -----------------------------


@torch.no_grad()
def decode_best_span_topk(start_logits, end_logits, max_answer_len=15, topk=20, length_penalty=0.02):
    L = start_logits.size(0)
    k = min(topk, L)

    s_vals, s_idx = torch.topk(start_logits, k=k)
    e_vals, e_idx = torch.topk(end_logits, k=k)

    best_score = -1e30
    best_s, best_e = 0, 0

    for i in range(k):
        s = s_idx[i].item()
        for j in range(k):
            e = e_idx[j].item()
            if e < s:
                continue
            span_len = e - s + 1
            if span_len > max_answer_len:
                continue

            score = s_vals[i].item() + e_vals[j].item() - length_penalty * span_len
            if score > best_score:
                best_score = score
                best_s, best_e = s, e

    return best_s, best_e


@torch.no_grad()
def predict_bidaf(model, ex, cfg):
    model.eval()

    model_vocab_size = cfg.get("model_vocab_size")

    ctx_ids = torch.LongTensor([
        encode(ex["context_tokens"], vocab, MAX_CONTEXT_LEN, add_cls=True, vocab_limit=model_vocab_size)
    ]).to(device)
    qst_ids = torch.LongTensor([
        encode(ex["question_tokens"], vocab, MAX_QUESTION_LEN, add_cls=False, vocab_limit=model_vocab_size)
    ]).to(device)

    s_logits, e_logits, na_logit = model(ctx_ids, qst_ids)
    s_logits = s_logits[0]
    e_logits = e_logits[0]

    p_no = torch.sigmoid(na_logit[0]).item()
    if p_no >= cfg["no_answer_threshold"]:
        return ""

    s_idx, e_idx = decode_best_span_topk(
        s_logits,
        e_logits,
        max_answer_len=cfg["max_answer_len"],
        topk=cfg["topk"],
        length_penalty=cfg["length_penalty"],
    )

    if s_idx == 0 or e_idx == 0:
        return ""

    s_txt = s_idx - 1
    e_txt = min(e_idx - 1, len(ex["context_tokens"]) - 1)
    if e_txt < s_txt:
        return ""

    pred = " ".join(ex["context_tokens"][s_txt:e_txt + 1])
    return clean_prediction(pred)


def _load_state_dict_any(path):
    raw = torch.load(path, map_location="cpu")
    if isinstance(raw, dict):
        if "state_dict" in raw and isinstance(raw["state_dict"], dict):
            raw = raw["state_dict"]
        if "model_state_dict" in raw and isinstance(raw["model_state_dict"], dict):
            raw = raw["model_state_dict"]
    if not isinstance(raw, dict):
        raise RuntimeError(f"Checkpoint non supporte: {path}")

    keys = list(raw.keys())
    if keys and all(k.startswith("module.") for k in keys):
        raw = {k[len("module."):]: v for k, v in raw.items()}
    keys = list(raw.keys())
    if keys and all(k.startswith("model.") for k in keys):
        raw = {k[len("model."):]: v for k, v in raw.items()}

    return raw


def _infer_dims_from_state_dict(sd, fallback_embed=300, fallback_hidden=128):
    embed_dim = fallback_embed
    hidden_size = fallback_hidden
    vocab_size = VOCAB_SIZE

    if "embedding.weight" in sd and sd["embedding.weight"].ndim == 2:
        vocab_size = int(sd["embedding.weight"].shape[0])
        embed_dim = int(sd["embedding.weight"].shape[1])

    if "start_head.weight" in sd and sd["start_head.weight"].ndim == 2:
        hidden_size = int(sd["start_head.weight"].shape[1] // 2)

    return vocab_size, embed_dim, hidden_size


def load_bidaf_models():
    models_loaded = {}

    specs = [
        {
            "name": "v3",
            "klass": QAModelV3,
            "ckpt": "bidaf_model_v3_best.pt",
            "fallback_init": {"embed_dim": 200, "hidden_size": 128},
            "cfg": {"no_answer_threshold": 0.50, "topk": 20, "max_answer_len": 30, "length_penalty": 0.00},
        },
        {
            "name": "v4",
            "klass": QAModelV3,
            "ckpt": "bidaf_model_v4_best.pt",
            "fallback_init": {"embed_dim": 200, "hidden_size": 128},
            "cfg": {"no_answer_threshold": 0.10, "topk": 20, "max_answer_len": 15, "length_penalty": 0.02},
        },
        {
            "name": "v5",
            "klass": QAModelV5,
            "ckpt": "bidaf_model_v5_best.pt",
            "fallback_init": {"embed_dim": 300, "hidden_size": 192},
            "cfg": {"no_answer_threshold": 0.10, "topk": 20, "max_answer_len": 15, "length_penalty": 0.02},
            "cfg_path": "bidaf_model_v5_config.json",
        },
        {
            "name": "v6",
            "klass": QAModelV6,
            "ckpt": "bidaf_model_v6_best.pt",
            "fallback_init": {"embed_dim": 300, "hidden_size": 224},
            "cfg": {"no_answer_threshold": 0.05, "topk": 28, "max_answer_len": 20, "length_penalty": 0.015},
            "cfg_path": "bidaf_model_v6_config.json",
        },
    ]

    for s in specs:
        if not os.path.exists(s["ckpt"]):
            print(f"[skip] {s['name']} checkpoint manquant: {s['ckpt']}")
            continue

        cfg = dict(s["cfg"])
        if s.get("cfg_path") and os.path.exists(s["cfg_path"]):
            with open(s["cfg_path"], "r", encoding="utf-8") as f:
                cfg_json = json.load(f)
            cfg.update({
                "no_answer_threshold": float(cfg_json.get("no_answer_threshold", cfg["no_answer_threshold"])),
                "topk": int(cfg_json.get("topk", cfg["topk"])),
                "max_answer_len": int(cfg_json.get("max_answer_len", cfg["max_answer_len"])),
                "length_penalty": float(cfg_json.get("length_penalty", cfg["length_penalty"])),
            })

        try:
            sd = _load_state_dict_any(s["ckpt"])
            model_vocab_size, embed_dim, hidden_size = _infer_dims_from_state_dict(
                sd,
                fallback_embed=s["fallback_init"]["embed_dim"],
                fallback_hidden=s["fallback_init"]["hidden_size"],
            )

            model = s["klass"](
                vocab_size=model_vocab_size,
                embed_dim=embed_dim,
                hidden_size=hidden_size,
            ).to(device)
            model.load_state_dict(sd, strict=True)
            model.eval()

            cfg["model_vocab_size"] = int(model_vocab_size)
            models_loaded[s["name"]] = {"model": model, "cfg": cfg}
            print(
                f"[ok] {s['name']} charge depuis {s['ckpt']} "
                f"(vocab={model_vocab_size}, embed_dim={embed_dim}, hidden_size={hidden_size})"
            )
        except Exception as e:
            print(f"[skip] {s['name']} incompatible: {type(e).__name__}: {e}")

    return models_loaded


bidaf_models = load_bidaf_models()
print("BiDAF charges:", list(bidaf_models.keys()))



[ok] v3 charge depuis bidaf_model_v3_best.pt (vocab=80404, embed_dim=200, hidden_size=128)
[ok] v4 charge depuis bidaf_model_v4_best.pt (vocab=80404, embed_dim=200, hidden_size=128)
[ok] v5 charge depuis bidaf_model_v5_best.pt (vocab=80404, embed_dim=300, hidden_size=192)
[ok] v6 charge depuis bidaf_model_v6_best.pt (vocab=80404, embed_dim=300, hidden_size=224)
BiDAF charges: ['v3', 'v4', 'v5', 'v6']


In [65]:
# -----------------------------
# Prédicteur BERT (bert_qa_best)
# -----------------------------

BERT_DIR = "bert_qa_best"
bert_bundle = None

if os.path.isdir(BERT_DIR):
    bert_tokenizer = AutoTokenizer.from_pretrained(BERT_DIR, use_fast=True)
    bert_model = AutoModelForQuestionAnswering.from_pretrained(BERT_DIR).to(device)
    bert_model.eval()
    bert_bundle = {"tokenizer": bert_tokenizer, "model": bert_model}
    print(f"[ok] bert chargé depuis: {BERT_DIR}")
else:
    print(f"[skip] bert dossier manquant: {BERT_DIR}")


@torch.no_grad()
def predict_bert(ex, null_score_diff_threshold=0.0, n_best=20, max_answer_len=30, max_length=384, stride=128):
    if bert_bundle is None:
        return ""

    tokenizer = bert_bundle["tokenizer"]
    model = bert_bundle["model"]

    context = ex["context"]
    question = ex["question"]

    enc = tokenizer(
        question,
        context,
        truncation="only_second",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        return_tensors="pt",
        padding="max_length",
    )

    best_text = ""
    best_non_null_score = -1e9
    min_null_score = 1e9

    for i in range(enc["input_ids"].shape[0]):
        input_ids = enc["input_ids"][i].unsqueeze(0).to(device)
        attn = enc["attention_mask"][i].unsqueeze(0).to(device)
        offsets = enc["offset_mapping"][i]
        seq_ids = enc.sequence_ids(i)

        out = model(input_ids=input_ids, attention_mask=attn)
        s_logits = out.start_logits[0].detach().cpu()
        e_logits = out.end_logits[0].detach().cpu()

        null_score = (s_logits[0] + e_logits[0]).item()
        min_null_score = min(min_null_score, null_score)

        s_top = torch.topk(s_logits, k=min(n_best, s_logits.shape[0])).indices.tolist()
        e_top = torch.topk(e_logits, k=min(n_best, e_logits.shape[0])).indices.tolist()

        for s_idx in s_top:
            for e_idx in e_top:
                if e_idx < s_idx:
                    continue
                if e_idx - s_idx + 1 > max_answer_len:
                    continue
                if seq_ids[s_idx] != 1 or seq_ids[e_idx] != 1:
                    continue

                s_char, _ = offsets[s_idx].tolist()
                _, e_char = offsets[e_idx].tolist()
                if s_char == e_char:
                    continue

                score = (s_logits[s_idx] + e_logits[e_idx]).item()
                if score > best_non_null_score:
                    best_non_null_score = score
                    best_text = context[s_char:e_char].strip()

    if min_null_score - best_non_null_score > null_score_diff_threshold:
        return ""
    return clean_prediction(best_text)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2202.88it/s, Materializing param=qa_outputs.weight]                                      


[ok] bert chargé depuis: bert_qa_best


In [66]:
# -----------------------------
# Format simple demandé: mêmes exemples, modèle par modèle
# -----------------------------

N = min(5, len(dev_examples))


def _print_block(model_name, predictor_fn):
    print(f"\n### {model_name}")
    for i in range(N):
        ex = dev_examples[i]
        pred = predictor_fn(ex)
        print(f"Exemple {i+1}")
        print("Q:", " ".join(ex["question_tokens"]))
        print("True:", ex["truth"] if ex["truth"] else "[NO_ANSWER]")
        print("Pred:", pred if pred else "[NO_ANSWER]")
        print("-" * 90)


# v3 / v4 / v5 / v6 via predict_bidaf
for _name in ["v3", "v4", "v5", "v6"]:
    if _name in bidaf_models:
        _bundle = bidaf_models[_name]
        _print_block(
            _name,
            lambda ex, _m=_bundle["model"], _cfg=_bundle["cfg"]: predict_bidaf(_m, ex, _cfg),
        )
    else:
        print(f"\n### {_name}")
        print("[NOT_LOADED]")


# bert
if bert_bundle is not None:
    _print_block("bert", lambda ex: predict_bert(ex, null_score_diff_threshold=0.0))
else:
    print("\n### bert")
    print("[NOT_LOADED]")




### v3
Exemple 1
Q: in what country is normandy located ?
True: France
Pred: france
------------------------------------------------------------------------------------------
Exemple 2
Q: when were the normans in normandy ?
True: 10th and 11th centuries
Pred: 10th
------------------------------------------------------------------------------------------
Exemple 3
Q: from which countries did the norse originate ?
True: Denmark, Iceland and Norway
Pred: france
------------------------------------------------------------------------------------------
Exemple 4
Q: who was the norse leader ?
True: Rollo
Pred: charles iii of west francia
------------------------------------------------------------------------------------------
Exemple 5
Q: what century did the normans first gain their separate identity ?
True: 10th century
Pred: 10th
------------------------------------------------------------------------------------------

### v4
Exemple 1
Q: in what country is normandy located ?
True: Fra